# 02B Incremental Read + Append All vs Full Read + Append New Only

Compare two independent source-to-target paths across three days: target-scoped Incremental Read + Append All, then Full Read + Append New Only.


### Before you begin

Run 00B and then 00C to stage Day 1 in Bronze. Run this notebook once for Day 1. For Day 2, set `DEMO_DAY = 2` and rerun; the short staging cell adds four new movements. Day 3 has no new source records. Do not rerun 00C after advancing the scenario.


# 0. Workspace — Setup Notebook

Run `setup_notebook()` to load `00_env_config`, resolve the caller environment, validate targets, and display configuration readiness. For shared configuration, pass `config_workspace_id` here. Setup Pipeline reuses this configuration.

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    finalize_pipeline,
    orchestrate_read,
    orchestrate_write,
    profile_table,
    read_lakehouse_csv,
    read_lakehouse_table,
    resolve_table_id,
    setup_notebook,
    setup_pipeline,
    write_lakehouse_table,
)


context = setup_notebook()

# For shared config, use the workspace hosting 00_env_config:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## Demo day

00C prepares Day 1. After a successful run, change to `2` and rerun for the new batch, then `3` for the no-change run. Never return to Day 1 after advancing Bronze.


In [ ]:
DEMO_DAY = 1

if DEMO_DAY not in {1, 2, 3}:
    raise ValueError("DEMO_DAY must be 1, 2, or 3.")


## Prepare the Day 2 Bronze input

00C already staged Day 1. This is demo input preparation, not incremental pipeline logic. Day 3 changes nothing.

This is explicit demo source preparation before pipeline selection. It changes Bronze independently of the later pipeline execution; the orchestrated Reads and Writes use the selected mode. Run it deliberately when advancing the demo day.


In [ ]:
if DEMO_DAY == 2:
    day2_df = read_lakehouse_csv(
        "demo/inventory_day2.csv", store="Bronze",
        header=True, inferSchema=True,
    )
    existing_df = read_lakehouse_table(
        table_name="inventory_movements", store="Bronze", schema="demo",
    )
    # Replaying Day 2 keeps each movement_id once in the Bronze source.
    updated_df = existing_df.unionByName(day2_df).dropDuplicates(["movement_id"])
    write_lakehouse_table(
        updated_df, table_name="inventory_movements", store="Bronze", schema="demo", mode="overwrite",
    )
    print("Bronze inventory rows:", updated_df.count())


# 1. Configure the two pipeline executions

Use **Dry Run** to preview without writing, **Ungoverned** to run without Data Contracts, or **Governed** with approved activations and Contracts. Pipeline source participation and reporting are automatic.

The active execution accepts only one Read for a logical source. The notebook therefore runs and finalizes Path A before starting Path B. This keeps Path A's Source Observation and accepted checkpoint associated only with its Append All target. Dry Run returns session-only summaries; persistent modes record the actual outcomes.


In [ ]:
PIPELINE_MODE = "Ungoverned"  # Dry Run, Ungoverned, Governed

pipelines = {}
pipeline_results = {}
sources = {}
writes = {}


# 2. Path A — Incremental Read + Append All

Resolve the Append All target's canonical identity before the Read. `modified_datetime` advances a checkpoint scoped to this exact Bronze source and Silver target. Day 1 bootstraps all 8 rows, Day 2 reads 4, and Day 3 returns an empty scope.


In [ ]:
pipelines["incremental_append_all"] = setup_pipeline(
    pipeline_mode=PIPELINE_MODE,
    source_drift_enabled=False,  # Opt in to complete-source observation.
    source_drift_interval_days=1,  # Since confirmed shared source baseline acceptance.
    source_drift_block=True,  # False warns; baseline acceptance still waits for Finalize.
)


## READ 1 — Inventory movements


In [ ]:
append_all_target_id = resolve_table_id(
    store="Silver",
    schema="demo",
    table_name="inventory_movements_append_all",
)

source = orchestrate_read(

    store="Bronze",
    schema="demo",
    table_name="inventory_movements",
    read_mode="incremental",
    query=None,

    read_parameters={"watermark_column": "modified_datetime"},
    target_table_id=append_all_target_id,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["incremental_append_all"] = source

# Inspect the actual rows returned by this Read.
display(sources["incremental_append_all"]["dataframe"].limit(10))


# 3. Apply the shared deterministic transformation


**Optional — Profile your transformation:** FabricOps already profiles governed Reads and Writes according to pipeline settings. Uncomment the example below to inspect a transformed DataFrame before writing. DataFrame-only profiling does not persist governed profile metadata; grain discovery and frequency profiling are off by default.


In [ ]:
def transform_inventory(dataframe):
    """Apply the shared deterministic inventory business transformation."""
    return (
        dataframe
        .withColumn("movement_date", F.to_date("modified_datetime"))
        .withColumn("quantity_in", F.greatest(F.col("quantity_change"), F.lit(0)))
        .withColumn("quantity_out", F.greatest(-F.col("quantity_change"), F.lit(0)))
    )


incremental_append_all_df = transform_inventory(
    sources["incremental_append_all"]["dataframe"]
)

# Optional development inspection.
# display(incremental_append_all_df)

# Optional: profile the transformed DataFrame before writing (no metadata persistence).
# transformed_profile = profile_table(dataframe=incremental_append_all_df)


# 4. Publish Path A and finalize its checkpoint

Append All inserts every row submitted by the incremental scope. FabricOps accepts checkpoint progress only after successful publication. Always call the Write on Day 3 as well; FabricOps owns empty-scope handling.

Append All does not deduplicate. Its safety depends on target-specific incremental progress and verified publication recovery; replaying previously consumed rows as new work can duplicate them.


In [ ]:
# Path A's Read is the only source registered in this execution.


## Write the incremental scope with Append All


In [ ]:
writes["inventory_movements_append_all"] = orchestrate_write(
    incremental_append_all_df,

    store="Silver",
    schema="demo",
    table_name="inventory_movements_append_all",
    write_mode="append_all",

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the final prepared input submitted to the Write operation. This is not a destination read-back.
write_input = writes["inventory_movements_append_all"]["dataframe"]
display(write_input.limit(10))

# Release the caller-owned inspection snapshot when finished.
write_input.unpersist()

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["inventory_movements_append_all"].get("dq_result"))
# display((writes["inventory_movements_append_all"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the final published target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="inventory_movements_append_all").orderBy("movement_id").limit(10))


## Finalize Path A before starting another Read of the source


In [ ]:
pipeline_results["incremental_append_all"] = finalize_pipeline(
    pipelines["incremental_append_all"]
)


# 5. Path B — Full Read + Append New Only

Start a fresh pipeline execution, read the complete Bronze source, and apply the same transformation to this separate DataFrame. Append New Only compares final `_row_hash` values with the target and inserts only unseen rows.


In [ ]:
pipelines["full_append_new_only"] = setup_pipeline(
    pipeline_mode=PIPELINE_MODE,
    source_drift_enabled=False,  # Opt in to complete-source observation.
    source_drift_interval_days=1,  # Since confirmed shared source baseline acceptance.
    source_drift_block=True,  # False warns; baseline acceptance still waits for Finalize.
)


## READ 2 — Inventory movements


In [ ]:
source = orchestrate_read(

    store="Bronze",
    schema="demo",
    table_name="inventory_movements",
    read_mode="full",
    query=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["full_append_new_only"] = source

# Inspect the actual rows returned by this Read.
display(sources["full_append_new_only"]["dataframe"].limit(10))


## TRANSFORM 2 — Apply the shared transformation


In [ ]:
full_append_new_only_df = transform_inventory(
    sources["full_append_new_only"]["dataframe"]
)

# Optional development inspection.
# display(full_append_new_only_df)

# Optional: profile the transformed DataFrame before writing (no metadata persistence).
# transformed_profile = profile_table(dataframe=full_append_new_only_df)


## Write the full scope with Append New Only


In [ ]:
writes["inventory_movements_append_new_only"] = orchestrate_write(
    full_append_new_only_df,

    store="Silver",
    schema="demo",
    table_name="inventory_movements_append_new_only",
    write_mode="append_new_only",

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the final prepared input submitted to the Write operation, including FabricOps audit and processing columns.
# Append New Only contains only previously unseen, deduplicated rows; this is not a destination read-back.
write_input = writes["inventory_movements_append_new_only"]["dataframe"]
display(write_input.limit(10))

# Release the caller-owned inspection snapshot when finished.
write_input.unpersist()

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["inventory_movements_append_new_only"].get("dq_result"))
# display((writes["inventory_movements_append_new_only"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the final published target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="inventory_movements_append_new_only").orderBy("movement_id").limit(10))


In [ ]:
pipeline_results["full_append_new_only"] = finalize_pipeline(
    pipelines["full_append_new_only"]
)


## Expected results and repeat runs

Run both independently finalized paths once per day, in order, starting with fresh Silver targets. Dry Run evaluates input without publishing and cannot demonstrate persisted counts. Rows submitted are the prepared business rows passed to Write; rows committed are the business rows newly persisted by that Write.

| Day | Incremental rows read / submitted / committed | Full rows read / submitted / committed | Append All total | Append New Only total |
|---|---:|---:|---:|---:|
| 1 | 8 / 8 / 8 | 8 / 8 / 8 | 8 | 8 |
| 2 | 4 / 4 / 4 | 12 / 12 / 4 | 12 | 12 |
| 3 | 0 / 0 / 0 | 12 / 12 / 0 | 12 | 12 |

Day 1 is a full incremental bootstrap because no accepted source-to-target checkpoint exists. Day 2 Incremental Read retrieves only four new movements. Day 3 Incremental Read returns an empty scope, while the independent Full Read still returns and submits all 12 rows; Append New Only commits zero new business rows.

### Choose the strategy

- **Incremental Read + Append All:** suitable for large sources with reliable watermarks or change tracking. Reads newly eligible records and avoids repeated complete processing scans. Correct checkpoints and replay handling are essential; replaying consumed records as new work can duplicate them. Full Source Observation or profiling, when due, can still scan the complete source.
- **Full Read + Append New Only:** suitable when duplicate prevention matters or reliable incremental extraction is unavailable. Reads the complete source every time and inserts only unseen row hashes. The cost is full source scanning and distributed hash comparison. Unchanged data causes no additional business rows.
Hashes cover final values and logical types after transformation and Sensitive Data treatment. Source timestamps and Job IDs participate; FabricOps audit and SCD lifecycle fields are excluded. Avoid adding volatile values to this experiment. Append New Only does not update keys, infer deletions or remove existing duplicates. Use SCD1 for current truth or SCD2 for history.

For a complete lifecycle restart, stop current and earlier demo jobs and run the single 00C reset cell with `RESET_DEMO = True`. This clears all metadata and native lifecycle artifacts in the dedicated `demo_metadata` item, restores five Day 1 sources and removes six outputs. Wait for `RESET COMPLETE` before Step 1, then set both 02B and 02C to `DEMO_DAY = 1`. Never reset between days or delete checkpoints separately.


# 6. Both paths are finalized

Each path finalized immediately after its Write. Inspect `pipeline_results` for separate execution summaries. A stopped path remains incomplete; do not start a new execution to conceal a failed publication.

In [ ]:
pipeline_results